<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 6: Titanic Kaggle Yarışması

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 6 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_titanic_kaggle.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_titanic_kaggle.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta06_siniflandirma_kaggle.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/06/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Kaggle Titanic yarışması submission
> - Feature Engineering stratejileri
> - Cross Validation

# Hafta 6 — Titanic Kaggle Yarışması

Bu defterde Kaggle'ın ünlü **Titanic: Machine Learning from Disaster** yarışmasını ele alacağız.

## İçindekiler
1. Veri Setini Oluşturma ve İnceleme
2. Veri Temizleme (Eksik Değerler)
3. Özellik Mühendisliği
4. Model Eğitimi: Lojistik Regresyon
5. Model Değerlendirme
6. Kaggle Gönderim Dosyası Oluşturma
7. Kaggle'a Gönderim Talimatları

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("Kütüphaneler yüklendi!")

## 1. Gerçek Kaggle Titanic Verisini Yükleme

Kaggle'ın ünlü Titanic yarışmasının **gerçek** eğitim verisini yüklüyoruz. Bu veri seti 891 yolcunun bilgilerini ve hayatta kalıp kalmadıklarını içerir.

**Kaynak:** [Kaggle Titanic Competition](https://www.kaggle.com/competitions/titanic)

**Gerçek eksik veriler:**
- Age: ~%20 eksik (177 kayıt)
- Cabin: ~%77 eksik (687 kayıt) 
- Embarked: 2 kayıt eksik

In [ ]:
# Kaggle Titanic gerçek eğitim verisini yükle
train_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
train = pd.read_csv(train_url)

print(f"Veri seti boyutu: {train.shape}")
print(f"\nSütunlar: {list(train.columns)}")
print(f"\nHayatta kalma oranı: {train['Survived'].mean():.2%}")
print(f"\nEksik veriler:")
print(train.isnull().sum()[train.isnull().sum() > 0])
train.head()

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
# Genel bilgi
print("Veri Seti Bilgisi")
print("=" * 40)
print(train.info())
print("\nTemel İstatistikler:")
train.describe()

### Veri Ön İşleme

Modeli eğitmeden önce veriyi temizliyoruz ve dönüştürüyoruz. Eksik değerleri doldurma, kategorik değişkenleri sayıya çevirme ve ölçeklendirme bu adımın parçasıdır.

In [ ]:
# Eksik değerler
print("Eksik Değerler:")
print(train.isnull().sum())

# Hayatta kalma oranı
print(f"\nHayatta Kalma Oranı: {train['Survived'].mean():.2%}")

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Görselleştirmeler
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Hayatta kalma dağılımı
train['Survived'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Hayatta Kalma Dağılımı')
axes[0, 0].set_xticklabels(['Hayatını Kaybetti (0)', 'Hayatta Kaldı (1)'], rotation=0)

# 2. Cinsiyete göre hayatta kalma
train.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0, 1], color=['#3498db', '#e91e63'])
axes[0, 1].set_title('Cinsiyete Göre Hayatta Kalma Oranı')
axes[0, 1].set_ylabel('Oran')
axes[0, 1].set_xticklabels(['Kadın', 'Erkek'], rotation=0)

# 3. Sınıfa göre hayatta kalma
train.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1, 0], color=['#f39c12', '#e67e22', '#d35400'])
axes[1, 0].set_title('Yolcu Sınıfına Göre Hayatta Kalma Oranı')
axes[1, 0].set_ylabel('Oran')
axes[1, 0].set_xticklabels(['1. Sınıf', '2. Sınıf', '3. Sınıf'], rotation=0)

# 4. Yaş dağılımı
train['Age'].dropna().hist(bins=30, ax=axes[1, 1], color='#9b59b6', edgecolor='white')
axes[1, 1].set_title('Yaş Dağılımı')
axes[1, 1].set_xlabel('Yaş')

plt.tight_layout()
plt.show()

## 2. Veri Temizleme — Eksik Değerler

- **Age:** Eksik yaş değerlerini **medyan** ile dolduruyoruz
- **Embarked:** Eksik biniş limanını **mod (en sık)** ile dolduruyoruz

In [ ]:
# Eksik yaş değerlerini medyan ile doldur
age_median = train['Age'].median()
print(f"Yaş medyanı: {age_median}")
train['Age'].fillna(age_median, inplace=True)

# Eksik biniş limanını mod ile doldur
embarked_mode = train['Embarked'].mode()[0]
print(f"Biniş limanı modu: {embarked_mode}")
train['Embarked'].fillna(embarked_mode, inplace=True)

# Kontrol
print(f"\nKalan eksik değerler:")
print(train.isnull().sum())

## 3. Özellik Mühendisliği

Modelin kullanabileceği sayısal özellikler oluşturuyoruz:
- **Sex:** male → 0, female → 1
- **Embarked:** One-hot encoding (S, C, Q)
- **FamilySize:** SibSp + Parch + 1 (kendisi dahil)

In [ ]:
# Cinsiyet kodlama
train['Sex_encoded'] = train['Sex'].map({'male': 0, 'female': 1})

# Biniş limanı kodlama
train['Embarked_S'] = (train['Embarked'] == 'S').astype(int)
train['Embarked_C'] = (train['Embarked'] == 'C').astype(int)
train['Embarked_Q'] = (train['Embarked'] == 'Q').astype(int)

# Aile büyüklüğü
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1

print("Yeni özellikler eklendi!")
print(f"\nAile Büyüklüğü dağılımı:")
print(train['FamilySize'].value_counts().sort_index())

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Aile büyüklüğüne göre hayatta kalma
plt.figure(figsize=(10, 5))
train.groupby('FamilySize')['Survived'].mean().plot(kind='bar', color='#2196F3', edgecolor='white')
plt.title('Aile Büyüklüğüne Göre Hayatta Kalma Oranı')
plt.xlabel('Aile Büyüklüğü')
plt.ylabel('Hayatta Kalma Oranı')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Model Eğitimi — Lojistik Regresyon

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Özellik seçimi
features = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 
            'Fare', 'Embarked_S', 'Embarked_C', 'Embarked_Q', 'FamilySize']

X = train[features]
y = train['Survived']

print(f"Özellik sayısı: {len(features)}")
print(f"Özellikler: {features}")

# Eğitim ve doğrulama setlerine ayır
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nEğitim seti: {X_train.shape[0]} örnek")
print(f"Doğrulama seti: {X_val.shape[0]} örnek")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Lojistik Regresyon modeli
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Eğitim ve doğrulama doğruluğu
train_acc = model.score(X_train, y_train)
val_acc = model.score(X_val, y_val)

print(f"Eğitim Doğruluğu:    {train_acc:.4f}")
print(f"Doğrulama Doğruluğu: {val_acc:.4f}")

## 5. Model Değerlendirme

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Doğrulama seti tahminleri
y_pred = model.predict(X_val)

# Karışıklık matrisi
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Hayatını Kaybetti', 'Hayatta Kaldı'],
            yticklabels=['Hayatını Kaybetti', 'Hayatta Kaldı'])
plt.title('Karışıklık Matrisi — Doğrulama Seti')
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.tight_layout()
plt.show()

# Sınıflandırma raporu
print("\nSınıflandırma Raporu:")
print(classification_report(y_val, y_pred, 
                            target_names=['Hayatını Kaybetti', 'Hayatta Kaldı']))

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Özellik önem katsayıları
coef_df = pd.DataFrame({
    'Özellik': features,
    'Katsayı': model.coef_[0]
}).sort_values('Katsayı', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in coef_df['Katsayı']]
plt.barh(coef_df['Özellik'], coef_df['Katsayı'], color=colors)
plt.title('Lojistik Regresyon — Özellik Katsayıları')
plt.xlabel('Katsayı Değeri')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Kaggle Gönderim Dosyası Oluşturma

Kaggle yarışması için bir gönderim CSV dosyası oluşturuyoruz. Gerçek yarışmada test seti ayrı verilir, biz burada doğrulama setimizi kullanıyoruz.

In [ ]:
# Gönderim dosyası oluştur
submission = pd.DataFrame({
    'PassengerId': train.loc[X_val.index, 'PassengerId'].values,
    'Survived': y_pred
})

print("Gönderim dosyası önizlemesi:")
print(submission.head(10))
print(f"\nToplam tahmin: {len(submission)}")
print(f"Hayatta kalan tahmin oranı: {submission['Survived'].mean():.2%}")

# CSV kaydet
# submission.to_csv('titanic_submission.csv', index=False)
# print("\nDosya kaydedildi: titanic_submission.csv")

## 7. Kaggle'a Gönderim Talimatları

### Adım 1: Kaggle Hesabı
- [kaggle.com](https://www.kaggle.com) adresine gidin ve ücretsiz hesap oluşturun.

### Adım 2: Yarışmaya Katılın
- [Titanic Yarışması](https://www.kaggle.com/c/titanic) sayfasına gidin.
- **"Join Competition"** butonuna tıklayın ve kuralları kabul edin.

### Adım 3: Veri İndirin
- Yarışma sayfasındaki **"Data"** sekmesinden `train.csv` ve `test.csv` dosyalarını indirin.
- `train.csv` ile modelinizi eğitin, `test.csv` için tahmin yapın.

### Adım 4: Gönderim Dosyasını Hazırlayın
- CSV dosyası **sadece iki sütun** içermelidir: `PassengerId` ve `Survived`
- `Survived` sütunu 0 veya 1 değerleri almalıdır.

```csv
PassengerId,Survived
892,0
893,1
894,0
...
```

### Adım 5: Gönderin
- **"Submit Predictions"** butonuna tıklayın.
- CSV dosyanızı yükleyin.
- Skorunuz hesaplanacak ve sıralama tablosunda yerinizi göreceksiniz.

### Kaggle CLI ile Gönderim (İsteğe Bağlı)
```bash
pip install kaggle
kaggle competitions submit -c titanic -f titanic_submission.csv -m "İlk denemem"
```

### İyileştirme Önerileri
- Farklı algoritmalar deneyin (Random Forest, XGBoost)
- Daha fazla özellik mühendisliği yapın (isimlerden unvan çıkarma, yaş grupları)
- Hiperparametre optimizasyonu yapın (GridSearchCV)
- Ensemble yöntemler kullanın

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>